# 05 The cascade as a state machine

The encoder decides, the conformal gate checks, and an LLM sees only the messages the gate refuses. Control flow comes from the **state**, never from model text:
the LLM's reply is validated against the closed route list in code, and a reply outside the list goes to a person.

**What is real and what is replayed.** The slow tier here is Claude Haiku. Its answers for the 200 test messages were produced once, in a Claude Code session, from
the route descriptions alone (it never saw the labels), and recorded in `data/llm/test_haiku_answers.jsonl`. The escalation node below looks the recorded answer up by message id.
No API call is made in this notebook, so **no LLM latency or price is measured here**; the notebook counts calls instead.

In [1]:
import json, hashlib, time, sqlite3, os
import numpy as np
from pathlib import Path
from typing import TypedDict, Optional, Literal
from scipy.special import softmax
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
ENC, T, ALPHA = r03["encoder"], r03["T"], r03["alpha_chosen"]
Q = r03[f"conformal_alpha_{ALPHA}"]["q"]
print(f"encoder {ENC} | temperature {T:.2f} | conformal threshold q={Q:.3f} for alpha={ALPHA} | hardware: {m.hardware()}")

tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); _, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)
idx_of = {r["id"]: i for i, r in enumerate(te)}

idmap = json.loads(Path("data/llm/test_id_map.json").read_text())               # opaque id -> message id
llm_answers = {idmap[j["id"]]: j["route"] for j in map(json.loads, Path("data/llm/test_haiku_answers.jsonl").read_text().splitlines())}
print(f"recorded slow-tier answers: {len(llm_answers)}; outside the closed list: {sum(a not in m.ROUTES for a in llm_answers.values())}")

encoder bge-small | temperature 0.28 | conformal threshold q=0.849 for alpha=0.02 | hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0
recorded slow-tier answers: 200; outside the closed list: 0


**Reading the output.** The notebook loads the choices made earlier: encoder `bge-small`, temperature 0.28 and conformal threshold `q` = 0.849 for alpha 0.02 (notebook 03). The 200 recorded slow-tier answers are all inside the closed list of eight routes.

## The state
One typed record travels through the graph. Every field is written by exactly one node.

In [2]:
class State(TypedDict, total=False):
    message_id: str
    text: str
    route_set: list[str]                         # conformal prediction set, set by `gate`
    decision: Literal["act", "escalate", "human"]
    route: Optional[str]                         # final route, set by act / escalate / review
    handled_by: Literal["encoder", "llm", "person"]
    status: str                                  # set by `execute`
    idem_key: str

DECISION_LOGIC_VERSION = "gate-v1:alpha=0.02"   # bump when the gate, the model or the thresholds change

## Nodes: classify and gate
`classify` is one encoder forward pass plus the head. `gate` turns the probabilities into a set and the set into a decision: one route acts, several escalate, none goes to a person.

In [3]:
loaded = m.load_encoder(ENC)
def classify(state: State) -> State:
    x = m.encode(ENC, [state["text"]], _loaded=loaded)
    p = softmax(clf.decision_function(x) / T, axis=1)[0]
    return {"route_set": [m.ROUTES[i] for i in np.where(p >= 1 - Q)[0]]}

def gate(state: State) -> State:
    s = state["route_set"]
    if len(s) == 1: return {"decision": "act", "route": s[0], "handled_by": "encoder"}
    return {"decision": "escalate" if len(s) > 1 else "human"}

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The escalation node: a closed reply, validated in code
The recorded reply must be one of the eight route names. Anything else, or no recorded reply, routes to a person. The type guarantees the format; whether the route is right is measured below.

In [4]:
LLM_CALLS = []     # one entry per slow-tier call, so the scorecard can count them

def escalate(state: State) -> State:
    LLM_CALLS.append(state["message_id"])
    reply = llm_answers.get(state["message_id"])
    if reply in m.ROUTES: return {"route": reply, "handled_by": "llm"}
    return {"decision": "human"}

def review(state: State) -> State:
    # Pauses the graph and persists its state. The caller resumes it with a person's decision.
    answer = interrupt({"message_id": state["message_id"], "text": state["text"], "candidates": state.get("route_set", [])})
    return {"route": answer, "handled_by": "person"}

## Idempotent execute
The side effect (here: writing the message to a queue) must happen at most once per message and decision logic. The key is a hash of the message id, the decision-logic version
and the action. A caller first claims the key with one atomic insert; only the caller that wins the claim writes. A second caller finds either a finished result, which it returns
without writing, or a claim still in progress, which it reports so that its own caller retries later. Stripe's API behaves the same way for a retry that arrives while the first request is still running: it does not run it a second time.

**What the next cell does, step by step.**

1. The key is the first 16 hex characters of a SHA-256 hash of the message id, the decision-logic version and the action.
2. `INSERT OR IGNORE` tries to write the key with the status `in_progress`. The database makes this atomic: one caller gets `rowcount == 1` and wins the claim, every other caller gets 0.
3. A caller that lost reads the status. `in_progress` means the winner is still working, so it reports that; anything else means a finished result, which it returns as `replayed:...` without writing.
4. The winner does the side effect, the queue write, outside the lock so a slow write does not block other callers, then records the row and marks the key `written`.
5. If the write raises, the claim is deleted so a retry can run.

`db_lock` exists only because SQLite's in-memory connection is shared between threads.

In [5]:
import threading
db = sqlite3.connect(":memory:", check_same_thread=False)
db_lock = threading.Lock()          # one connection shared by threads: serialize access to it (the claim itself is atomic in SQL)
db.execute("CREATE TABLE done (idem_key TEXT PRIMARY KEY, status TEXT)")
db.execute("CREATE TABLE queue_writes (message_id TEXT, route TEXT)")
SLOW_WRITE_S = 0.0                  # raised in the concurrency demo to make the write take time, as a real queue call does

def execute(state: State) -> State:
    action = f"queue:{state['route']}"
    key = hashlib.sha256(f"{state['message_id']}|{DECISION_LOGIC_VERSION}|{action}".encode()).hexdigest()[:16]
    with db_lock:
        claimed = db.execute("INSERT OR IGNORE INTO done VALUES (?, 'in_progress')", (key,)).rowcount == 1
        if not claimed:
            status = db.execute("SELECT status FROM done WHERE idem_key=?", (key,)).fetchone()[0]
            return {"idem_key": key, "status": "in_progress" if status == "in_progress" else f"replayed:{status}"}
    try:
        time.sleep(SLOW_WRITE_S)                                  # the side effect happens outside the lock
        with db_lock:
            db.execute("INSERT INTO queue_writes VALUES (?,?)", (state["message_id"], state["route"]))
            db.execute("UPDATE done SET status='written' WHERE idem_key=?", (key,))
    except Exception:
        with db_lock: db.execute("DELETE FROM done WHERE idem_key=?", (key,))   # release the claim so a retry can run
        raise
    return {"idem_key": key, "status": "written"}

## Wire the graph
Edges are chosen by `state["decision"]`, a field the code set, never by parsing model text.

In [6]:
def build(checkpointer=None):
    g = StateGraph(State)
    for name, fn in [("classify", classify), ("gate", gate), ("escalate", escalate), ("review", review), ("execute", execute)]:
        g.add_node(name, fn)
    g.add_edge(START, "classify"); g.add_edge("classify", "gate")
    g.add_conditional_edges("gate", lambda s: s["decision"], {"act": "execute", "escalate": "escalate", "human": "review"})
    g.add_conditional_edges("escalate", lambda s: s["decision"] if s.get("decision") == "human" else "act", {"act": "execute", "human": "review"})
    g.add_edge("review", "execute"); g.add_edge("execute", END)
    return g.compile(checkpointer=checkpointer)

app = build(MemorySaver())

### One message that acts, run twice

In [7]:
r = te[0]
cfg = {"configurable": {"thread_id": "demo-1"}}
out1 = app.invoke({"message_id": r["id"], "text": r["text"]}, cfg)
print("first run :", out1["decision"], "->", out1["route"], "|", out1["status"], "| key", out1["idem_key"])
out2 = app.invoke({"message_id": r["id"], "text": r["text"]}, {"configurable": {"thread_id": "demo-2"}})
print("second run:", out2["decision"], "->", out2["route"], "|", out2["status"], "| key", out2["idem_key"])
print("rows in queue_writes for this message:", db.execute("SELECT COUNT(*) FROM queue_writes WHERE message_id=?", (r["id"],)).fetchone()[0])

first run : act -> baggage_claim | written | key e383172571dc608c
second run: act -> baggage_claim | replayed:written | key e383172571dc608c
rows in queue_writes for this message: 1


**Reading the output.** Both runs use different thread ids, so LangGraph's own checkpoint does not connect them. The second run still produces the same key (`e383172571dc608c`), finds the first run's result and replays it, and the queue holds one row for the message.

### The same message from eight callers at once
A retry storm or a double-clicked button sends the same request several times before the first has finished. The first version below checks for a finished result and then writes, which lets
every caller through while the first write is still running; the claim-first `execute` lets exactly one through.

In [8]:
from concurrent.futures import ThreadPoolExecutor
SLOW_WRITE_S = 0.05
db.execute("CREATE TABLE naive_done (idem_key TEXT PRIMARY KEY)"); db.execute("CREATE TABLE naive_writes (message_id TEXT)")

def execute_check_then_write(state):
    key = state["message_id"] + "|naive"
    with db_lock: seen = db.execute("SELECT 1 FROM naive_done WHERE idem_key=?", (key,)).fetchone()
    if seen: return "replayed"
    time.sleep(SLOW_WRITE_S)
    with db_lock:
        db.execute("INSERT INTO naive_writes VALUES (?)", (state["message_id"],))
        db.execute("INSERT OR IGNORE INTO naive_done VALUES (?)", (key,))
    return "written"

st = {"message_id": te[1]["id"], "route": m.ROUTES[int(yte[1])]}
with ThreadPoolExecutor(8) as ex: naive = list(ex.map(lambda _: execute_check_then_write(st), range(8)))
with ThreadPoolExecutor(8) as ex: claimed = [r["status"] for r in ex.map(lambda _: execute(st), range(8))]
SLOW_WRITE_S = 0.0
n_naive = db.execute("SELECT COUNT(*) FROM naive_writes").fetchone()[0]
n_claim = db.execute("SELECT COUNT(*) FROM queue_writes WHERE message_id=?", (st["message_id"],)).fetchone()[0]
from collections import Counter
print("check-then-write :", dict(Counter(naive)), "-> queue rows:", n_naive)
print("claim-first      :", dict(Counter(claimed)), "-> queue rows:", n_claim)

check-then-write : {'written': 8} -> queue rows: 8
claim-first      : {'written': 1, 'in_progress': 7} -> queue rows: 1


**Reading the output.** `SLOW_WRITE_S = 0.05` makes the write take 50 ms, so all eight callers pass the check before any of them finishes. Check-then-write lets all eight through: 8 writes and 8 queue rows. Claim-first lets one through: 1 `written`, 7 `in_progress`, 1 queue row. The seven callers that lost are told to retry later.

### One message that pauses for a person
At alpha = 0.02 no test message produces an empty set, so the gate never sends one straight to a person. The person path is still reached when the slow tier fails:
a timeout, a malformed reply, a reply outside the route list. Here we simulate that by removing one recorded reply, run the message until the graph pauses, then resume it with a decision.

In [9]:
def set_size(row):
    x = m.encode(ENC, [row["text"]], _loaded=loaded); p = softmax(clf.decision_function(x) / T, axis=1)[0]
    return int((p >= 1 - Q).sum())
sizes = [set_size(r) for r in te]
print("set-size histogram on the test split:", {s: sizes.count(s) for s in sorted(set(sizes))})
pick = next(r for r, s in zip(te, sizes) if s > 1)
saved = llm_answers.pop(pick["id"])                      # simulate: the slow tier returned nothing usable
cfg = {"configurable": {"thread_id": "demo-human"}}
paused = app.invoke({"message_id": pick["id"], "text": pick["text"]}, cfg)
print("candidates the gate kept:", paused["route_set"])
print("paused for review:", "__interrupt__" in paused, "| rows written so far:", db.execute("SELECT COUNT(*) FROM queue_writes WHERE message_id=?", (pick["id"],)).fetchone()[0])
done = app.invoke(Command(resume=paused["route_set"][0]), cfg)
print("resumed ->", done["route"], "| handled_by", done["handled_by"], "|", done["status"])
llm_answers[pick["id"]] = saved                          # restore the recorded reply

set-size histogram on the test split: {1: 183, 2: 15, 3: 2}
candidates the gate kept: ['cancellation_request', 'rebooking_request', 'refund_claim']
paused for review: True | rows written so far: 0
resumed -> cancellation_request | handled_by person | written


**Reading the output.** The set sizes on the test split are 183 messages with one route, 15 with two and 2 with three, so 17 messages go to the slow tier (the same 17 the scorecard counts below). The demo takes the first message with a set larger than one and removes its recorded reply, as if the slow tier had timed out. The graph pauses at `review` with nothing written (`paused for review: True`, 0 rows). Resuming with a person's choice finishes the run, writes one row and records `handled_by person`.

## Run the whole test split

In [10]:
LLM_CALLS.clear()
rows_out, t0 = [], time.perf_counter()
per_msg_ms = []
for r in te:
    cfg = {"configurable": {"thread_id": f"run-{r['id']}"}}
    t1 = time.perf_counter()
    o = app.invoke({"message_id": r["id"], "text": r["text"]}, cfg)
    per_msg_ms.append((time.perf_counter() - t1) * 1000)
    if "__interrupt__" in o:
        rows_out.append(dict(id=r["id"], route=None, by="person-queue", decision=o.get("decision", "human"))); continue
    rows_out.append(dict(id=r["id"], route=o["route"], by=o["handled_by"], decision=o["decision"]))
wall = time.perf_counter() - t0
pm = np.array(per_msg_ms)
print(f"{len(te)} messages through the graph in {wall:.1f} s on this CPU")
print(f"per message through the whole graph: median {np.median(pm):.0f} ms, p95 {np.percentile(pm, 95):.0f} ms (the escalation node is a dictionary lookup, so the slow tier's own latency is NOT in these numbers)")
print("load average (1 / 5 / 15 min):", [round(x, 1) for x in __import__('os').getloadavg()])

200 messages through the graph in 7.2 s on this CPU
per message through the whole graph: median 32 ms, p95 54 ms (the escalation node is a dictionary lookup, so the slow tier's own latency is NOT in these numbers)
load average (1 / 5 / 15 min): [6.0, 7.2, 6.5]


**Reading the output.** 200 messages go through the whole graph in 7.2 s, a median of 32 ms and a p95 of 54 ms per message. That covers the encoder pass, the gate, the dictionary lookup that stands in for the slow tier and the execute node, so it excludes any real slow-tier latency. The load average of 6 on 4 cores means these timings are noisy.

## The scorecard against an LLM-only router
Accuracy is measured on the messages the cascade answered itself (encoder or LLM). Messages sent to a person are reported as a share, not scored.
The LLM-only router is the same recorded Haiku reply for all 200 messages. Intervals: Wilson (assumes independent items) and a cluster bootstrap over scenarios.

In [11]:
y = {r["id"]: yte[i] for i, r in enumerate(te)}
cl = {r["id"]: f"{r['label']}:{r['scenario']}" for r in te}
answered = [o for o in rows_out if o["route"] is not None]
ok_c = np.array([m.ROUTES.index(o["route"]) == y[o["id"]] for o in answered])
cl_c = np.array([cl[o["id"]] for o in answered])
n_enc = sum(o["by"] == "encoder" for o in rows_out); n_llm = sum(o["by"] == "llm" for o in rows_out); n_per = sum(o["route"] is None or o["by"] == "person" for o in rows_out)
print(f"handled by encoder {n_enc} ({n_enc/len(te):.1%}) | escalated to LLM {n_llm} ({n_llm/len(te):.1%}) | sent to a person {n_per} ({n_per/len(te):.1%})")
lo, hi = m.wilson(int(ok_c.sum()), len(ok_c)); clo, chi = m.cluster_bootstrap_ci(ok_c.astype(float), cl_c)
print(f"cascade accuracy on answered ({len(ok_c)}): {ok_c.mean():.3f}  Wilson [{lo:.3f}, {hi:.3f}]  cluster [{clo:.3f}, {chi:.3f}]")
for who in ("encoder", "llm"):
    s = np.array([m.ROUTES.index(o["route"]) == y[o["id"]] for o in answered if o["by"] == who])
    if len(s): print(f"   of which {who:8s} n={len(s):3d} accuracy {s.mean():.3f}  Wilson [{m.wilson(int(s.sum()), len(s))[0]:.3f}, {m.wilson(int(s.sum()), len(s))[1]:.3f}]")

llm_ok = np.array([m.ROUTES.index(llm_answers[r["id"]]) == yte[i] if llm_answers.get(r["id"]) in m.ROUTES else False for i, r in enumerate(te)])
cl_all = np.array([cl[r["id"]] for r in te])
lo2, hi2 = m.wilson(int(llm_ok.sum()), len(llm_ok)); clo2, chi2 = m.cluster_bootstrap_ci(llm_ok.astype(float), cl_all)
print(f"LLM-only accuracy on all {len(te)}: {llm_ok.mean():.3f}  Wilson [{lo2:.3f}, {hi2:.3f}]  cluster [{clo2:.3f}, {chi2:.3f}]")
print(f"slow-tier calls per 1,000 messages: LLM-only 1000 | cascade {len(LLM_CALLS) / len(te) * 1000:.0f}")

handled by encoder 183 (91.5%) | escalated to LLM 17 (8.5%) | sent to a person 0 (0.0%)


cascade accuracy on answered (200): 0.975  Wilson [0.943, 0.989]  cluster [0.935, 1.000]
   of which encoder  n=183 accuracy 0.984  Wilson [0.953, 0.994]
   of which llm      n= 17 accuracy 0.882  Wilson [0.657, 0.967]


LLM-only accuracy on all 200: 0.950  Wilson [0.910, 0.973]  cluster [0.875, 1.000]
slow-tier calls per 1,000 messages: LLM-only 1000 | cascade 85


**Reading the output.** The encoder answers 183 messages (91.5%) with accuracy 0.984; the slow tier answers 17 (8.5%) with accuracy 0.882 (Wilson 0.657 to 0.967, only 17 messages). The cascade reaches 0.975 on the 200 it answered and the slow-tier-only router 0.950, with overlapping intervals, while making 85 slow-tier calls per 1,000 messages instead of 1,000.

### Macro-F1, and tokens at the slow tier
Macro-F1 averages the per-route F1, so a small route counts as much as a big one. Tokens stand in for cost: the slow tier is billed per input and output token at a rate you choose,
so we count tokens per 1,000 messages for each router and leave the price to you. The count uses the bge-small tokenizer as a proxy for the slow tier's own tokenizer, so treat it as an estimate.

In [12]:
from sklearn.metrics import f1_score
y_c = np.array([y[o["id"]] for o in answered]); p_c = np.array([m.ROUTES.index(o["route"]) for o in answered])
y_all = np.array([y[r["id"]] for r in te]); p_llm = np.array([m.ROUTES.index(llm_answers[r["id"]]) for r in te])
print(f"macro-F1 on answered messages: cascade {f1_score(y_c, p_c, average='macro'):.3f} | LLM-only on all 200 {f1_score(y_all, p_llm, average='macro'):.3f}")

DESCRIPTIONS_PROMPT = ("Choose exactly one route for the customer message from this closed list: " + "; ".join(m.ROUTES) +
                       ". Reply with the route name only.\nMessage: ")
tok = loaded[0]
n_tok = lambda text: len(tok(text, add_special_tokens=False)["input_ids"])
in_tokens = {r["id"]: n_tok(DESCRIPTIONS_PROMPT + r["text"]) for r in te}
out_tokens = 5     # a route name is a few tokens; an assumption, not measured
esc_ids = LLM_CALLS
t_cascade_in = sum(in_tokens[i] for i in esc_ids) / len(te) * 1000
t_llm_in = sum(in_tokens.values()) / len(te) * 1000
print(f"input tokens per 1,000 messages (estimate): LLM-only {t_llm_in:,.0f} | cascade {t_cascade_in:,.0f}")
print(f"output tokens per 1,000 messages (about {out_tokens} per reply, assumed): LLM-only {out_tokens * 1000:,} | cascade {out_tokens * len(esc_ids) / len(te) * 1000:,.0f}")
print("cost per 1,000 messages = input tokens / 1e6 x your input price + output tokens / 1e6 x your output price")

macro-F1 on answered messages: cascade 0.975 | LLM-only on all 200 0.949
input tokens per 1,000 messages (estimate): LLM-only 95,160 | cascade 8,185
output tokens per 1,000 messages (about 5 per reply, assumed): LLM-only 5,000 | cascade 425
cost per 1,000 messages = input tokens / 1e6 x your input price + output tokens / 1e6 x your output price


**Reading the output.** Macro-F1 is 0.975 for the cascade and 0.949 for the slow-tier-only router. Input tokens per 1,000 messages fall from about 95,160 to about 8,185. The estimate uses the bge-small tokenizer and assumes 5 output tokens per reply, so multiply by your own prices and treat the result as an estimate.

### A local slow tier, timed
The recorded Haiku replies cost no time here, so the scorecard has no slow-tier latency. To put a number on it, a small local LLM (Qwen3-1.7B, in float32 on this CPU) answers the escalated messages plus a
sample of others. It is given the eight route names with their descriptions and asked for the team name only; we read the scores of the first token of each name in a single forward pass (the eight first tokens are distinct), so the answer is a choice from the closed list, as it is for the encoder.
This is **a local model, not Haiku**: its answers are not used in the scorecard above. It tells you what a slow tier costs in seconds on a CPU, and how it does on the hard slice.

A first version labeled the routes A to H and read the eight letters. On a run of this notebook it scored 0.18 on the 17 escalated messages and 0.22 on 23 others, where guessing scores 0.125: a 1.7B model picks a letter by position, not by meaning. That result is a recorded observation, not recomputed here.

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
LOCAL = "Qwen/Qwen3-1.7B"
DESC = {"baggage_claim": "luggage lost, delayed, damaged or missing; wants it traced or compensated",
        "cancellation_request": "asks to cancel a booking, flight, hotel or trip",
        "out_of_scope": "not about a booking or travel problem: chat, other businesses, nothing an agent can act on",
        "payment_dispute": "disputes a charge, double payment, unrecognized transaction or wrong amount",
        "rebooking_request": "wants to change dates, times or flights, or be moved to another flight",
        "refund_claim": "asks for money back for something paid: cancelled service, missed flight, fee",
        "service_complaint": "complains about rudeness, poor service or a bad experience, asking for no specific action",
        "visa_entry_question": "asks about visa, passport, vaccination or entry requirements"}
tokq = AutoTokenizer.from_pretrained(LOCAL)
t_load = time.perf_counter()
lm = AutoModelForCausalLM.from_pretrained(LOCAL, dtype=torch.float32).eval()
t_load = time.perf_counter() - t_load
first_ids = [tokq.encode(r, add_special_tokens=False)[0] for r in m.ROUTES]
assert len(set(first_ids)) == len(m.ROUTES), "route names must start with distinct tokens"
menu = "\n".join(f"- {r}: {DESC[r]}" for r in m.ROUTES)

def ask_local(text):
    msgs = [{"role": "user", "content": f"Route this customer message from a travel company to exactly one team.\nTeams:\n{menu}\n\nMessage: {text}\n\nAnswer with the team name only."}]
    ids = tokq.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt")
    with torch.inference_mode():
        logits = lm(input_ids=ids).logits[0, -1, first_ids]
    return int(logits.argmax()), int(ids.shape[1])

rng_s = np.random.default_rng(m.SEED)
others = [r["id"] for r in te if r["id"] not in set(esc_ids)]
sample_ids = list(esc_ids) + list(rng_s.choice(others, size=min(23, len(others)), replace=False))
by_id = {r["id"]: r for r in te}
ask_local(te[0]["text"])      # warm-up
local_ms, local_pred, local_len = [], {}, []
for i in sample_ids:
    t = time.perf_counter(); pred, n_in = ask_local(by_id[i]["text"]); local_ms.append((time.perf_counter() - t) * 1000)
    local_pred[i] = pred; local_len.append(n_in)
acc_esc_local = float(np.mean([local_pred[i] == y[i] for i in esc_ids])) if esc_ids else float("nan")
acc_esc_haiku = float(np.mean([m.ROUTES.index(llm_answers[i]) == y[i] for i in esc_ids])) if esc_ids else float("nan")
oth = [i for i in sample_ids if i not in set(esc_ids)]
acc_oth_local = float(np.mean([local_pred[i] == y[i] for i in oth]))
print(f"{LOCAL}: loaded in {t_load:.1f} s; {len(sample_ids)} messages; prompt {int(np.median(local_len))} tokens")
print(f"latency per message: median {np.median(local_ms):.0f} ms, p95 {np.percentile(local_ms, 95):.0f} ms (load average {os.getloadavg()[0]:.1f})")
print(f"on the {len(esc_ids)} escalated messages: local model {acc_esc_local:.2f}, recorded Haiku {acc_esc_haiku:.2f}")
print(f"on {len(oth)} other messages: local model {acc_oth_local:.2f}")
local_slow = dict(model=LOCAL, n=len(sample_ids), n_escalated=len(esc_ids), median_ms=float(np.median(local_ms)), p95_ms=float(np.percentile(local_ms, 95)),
                  load_average=list(os.getloadavg()), acc_escalated_local=acc_esc_local, acc_escalated_haiku=acc_esc_haiku, acc_other_local=acc_oth_local,
                  prompt_tokens=int(np.median(local_len)), load_seconds=t_load, letter_menu_first_attempt=dict(acc_escalated=0.176, acc_other=0.217))
del lm

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:21<00:21, 21.16s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:26<00:00, 11.88s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:26<00:00, 13.27s/it]

Qwen/Qwen3-1.7B: loaded in 27.2 s; 40 messages; prompt 233 tokens
latency per message: median 11484 ms, p95 26047 ms (load average 35.6)
on the 17 escalated messages: local model 0.65, recorded Haiku 0.88
on 23 other messages: local model 0.65


**Reading the output.** A local 1.7B model scores 0.65 on the 17 escalated messages, against 0.88 for the recorded replies, and 0.65 on 23 other messages. It took a median of 11,484 ms per message at a load average of 35.6, so the number is inflated by a busy machine. It still says that a slow tier on a CPU costs seconds per message where the graph above costs tens of milliseconds.

### Paired comparison on the messages the cascade answered
Exact McNemar test: among messages where exactly one system is right, is one system right more often than chance? Discordant counts, then the two-sided exact p-value.

In [14]:
llm_on_answered = np.array([llm_ok[idx_of[o["id"]]] for o in answered])
b = int(((ok_c) & (~llm_on_answered)).sum()); c = int(((~ok_c) & (llm_on_answered)).sum())
print(f"cascade right, LLM-only wrong: {b} | cascade wrong, LLM-only right: {c} | both right: {int((ok_c & llm_on_answered).sum())} | both wrong: {int((~ok_c & ~llm_on_answered).sum())}")
p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
print(f"exact McNemar two-sided p = {p:.3f}   (messages are variants of 40 scenarios, so treat this p-value as optimistic)")

res = dict(n=len(te), encoder=n_enc, llm=n_llm, person=n_per, cascade_acc=float(ok_c.mean()), cascade_wilson=[lo, hi], cascade_cluster=[clo, chi],
           llm_only_acc=float(llm_ok.mean()), llm_only_wilson=[lo2, hi2], llm_only_cluster=[clo2, chi2],
           llm_calls_per_1000_cascade=len(LLM_CALLS) / len(te) * 1000, mcnemar=dict(b=b, c=c, p=float(p)),
           graph_ms_median=float(np.median(pm)), graph_ms_p95=float(np.percentile(pm, 95)), load=list(__import__('os').getloadavg()),
           macro_f1_cascade=float(f1_score(y_c, p_c, average="macro")), macro_f1_llm_only=float(f1_score(y_all, p_llm, average="macro")),
           tokens_in_per_1000=dict(llm_only=t_llm_in, cascade=t_cascade_in), tokens_out_per_1000=dict(llm_only=out_tokens * 1000, cascade=out_tokens * len(esc_ids) / len(te) * 1000),
           local_slow_tier=local_slow, hardware=m.hardware(), decision_logic_version=DECISION_LOGIC_VERSION)
(RESULTS / "05_langgraph_cascade.json").write_text(json.dumps(res, indent=2, default=float))

cascade right, LLM-only wrong: 8 | cascade wrong, LLM-only right: 3 | both right: 187 | both wrong: 2
exact McNemar two-sided p = 0.227   (messages are variants of 40 scenarios, so treat this p-value as optimistic)


1547

**Reading the output.** Of the 200 messages, both systems are right on 187 and wrong on 2. The cascade is right and the slow-tier-only router wrong on 8; the reverse holds on 3. The exact p-value is 0.227, optimistic because the messages are variants of 40 scenarios, so the 2.5-point lead is not established.